# Card Reader

Loads the model we trained in Colab (`models/best.onnx`), moves the arm to a camera pose, takes a picture, and reads the cards.

**Before running:** put `best.onnx` from Colab into `HackRPI26/models/` on the Jetson.

In [ ]:
import json
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)
print("Poses:", list(poses))

## Load the model
Uses `onnxruntime` if it's installed, otherwise OpenCV's built-in DNN. Prints the output shape so we know the model loaded right.

In [ ]:
MODEL_PATH = 'models/best.onnx'

try:
    import onnxruntime as ort
    session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    BACKEND = 'onnxruntime'
except ImportError:
    net = cv2.dnn.readNetFromONNX(MODEL_PATH)
    BACKEND = 'opencv'
print("Backend:", BACKEND)

## Detection + blackjack helpers

In [ ]:
CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as training
IMGSZ = 416        # same size we trained/exported with
CONF = 0.5         # ignore detections less confident than this


def letterbox(frame, size=IMGSZ):
    """Resize keeping the shape, pad the rest with gray (same as YOLO training)."""
    h, w = frame.shape[:2]
    r = min(size / h, size / w)
    nw, nh = int(round(w * r)), int(round(h * r))
    pad_x, pad_y = (size - nw) / 2, (size - nh) / 2
    img = cv2.resize(frame, (nw, nh))
    img = cv2.copyMakeBorder(img, int(round(pad_y - 0.1)), int(round(pad_y + 0.1)),
                             int(round(pad_x - 0.1)), int(round(pad_x + 0.1)),
                             cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return img, r, pad_x, pad_y


def run_model(blob):
    """All model outputs. A segmentation model has two: predictions and mask 'prototypes'."""
    if BACKEND == 'onnxruntime':
        return session.run(None, {input_name: blob})
    net.setInput(blob)
    return net.forward(net.getUnconnectedOutLayersNames())


OUTLINES = {}   # card outline (pixels) for each box detect() returned last, when the model gives outlines


def detect(frame, conf=CONF, iou=0.5):
    """Returns a list of (label, confidence, (x1, y1, x2, y2)) in the original picture's pixels.
    Works with both models:
      - box model (old): overlapping duplicates are removed by comparing boxes
      - segmentation model (new): each card also has an outline; duplicates are removed by comparing OUTLINES,
        so two overlapping, turned cards (whose boxes cover each other) are both kept.
    Outlines of the returned cards go in OUTLINES[box]."""
    img, r, pad_x, pad_y = letterbox(frame)
    blob = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    pred, protos = None, None
    for o in run_model(blob):
        o = np.asarray(o)
        if o.ndim == 4:
            protos = o[0]                       # (32, mask h, mask w)
        elif o.ndim == 3:
            pred = o[0]
    nc = len(CLASSES)
    nm = protos.shape[0] if protos is not None else 0

    if pred.shape[0] > pred.shape[1] and pred.shape[1] in (6, 6 + nm):
        # End-to-end output (no NMS needed): rows = x1 y1 x2 y2 score class [mask coefficients]
        end2end = True
        keep = pred[:, 4] >= conf
        boxes, scores = pred[keep, :4], pred[keep, 4]
        ids, coefs = pred[keep, 5].astype(int), pred[keep, 6:6 + nm]
    else:
        # Standard output: (4 + classes [+ mask coefficients]) x N, boxes are cx cy w h
        end2end = False
        if pred.shape[0] != 4 + nc + nm:
            pred = pred.T
        cls_scores = pred[4:4 + nc]
        ids_all, s_all = cls_scores.argmax(0), cls_scores.max(0)
        keep = s_all >= conf
        cx, cy, w, h = pred[:4, keep]
        boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
        scores, ids, coefs = s_all[keep], ids_all[keep], pred[4 + nc:4 + nc + nm, keep].T

    order = np.argsort(-scores)[:100]
    boxes, scores, ids, coefs = boxes[order], scores[order], ids[order], coefs[order]

    masks = None
    if protos is not None and len(boxes):
        mh, mw = protos.shape[1:]
        m = 1 / (1 + np.exp(-(coefs @ protos.reshape(nm, -1))))
        masks = m.reshape(-1, mh, mw) > 0.5
        sx, sy = mw / IMGSZ, mh / IMGSZ
        for i, (x1, y1, x2, y2) in enumerate(boxes):    # keep each outline inside its own box
            inside = np.zeros((mh, mw), bool)
            inside[max(0, int(y1 * sy)):int(np.ceil(y2 * sy)), max(0, int(x1 * sx)):int(np.ceil(x2 * sx))] = True
            masks[i] &= inside

    def overlap(i, j):
        if masks is not None:
            inter = (masks[i] & masks[j]).sum()
            union = (masks[i] | masks[j]).sum()
            return inter / union if union else 0.0
        a, b = boxes[i], boxes[j]
        iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
        inter = iw * ih
        union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return inter / union if union else 0.0

    kept = []
    for i in range(len(boxes)):
        if end2end or all(overlap(i, j) <= iou for j in kept):
            kept.append(i)

    results = []
    OUTLINES.clear()
    H, W = frame.shape[:2]
    for i in kept:
        x1, y1, x2, y2 = boxes[i]
        # undo the letterbox: back to original picture pixels
        box = (int(np.clip((x1 - pad_x) / r, 0, W)), int(np.clip((y1 - pad_y) / r, 0, H)),
               int(np.clip((x2 - pad_x) / r, 0, W)), int(np.clip((y2 - pad_y) / r, 0, H)))
        results.append((CLASSES[ids[i]], float(scores[i]), box))
        if masks is not None and masks[i].any():
            big = cv2.resize(masks[i].astype(np.uint8), (IMGSZ, IMGSZ), interpolation=cv2.INTER_NEAREST)
            contours = cv2.findContours(big, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
            c = max(contours, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)
            OUTLINES[box] = ((c - [pad_x, pad_y]) / r).astype(np.int32)
    return results


def hand_value(labels):
    """Blackjack total. Aces count 11 unless that busts. DOWN cards are ignored."""
    total, aces = 0, 0
    for l in labels:
        if l == 'DOWN':
            continue
        if l == 'A':
            total += 11; aces += 1
        elif l in ('J', 'Q', 'K', '10'):
            total += 10
        else:
            total += int(l)
    while total > 21 and aces:
        total -= 10; aces -= 1
    return total, aces > 0   # (total, is_soft)

In [ ]:
# Quick check: run the model once on a blank picture
t = time.time()
img, _, _, _ = letterbox(np.zeros((480, 640, 3), np.uint8))
blob = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype(np.float32) / 255.0
outs = run_model(blob)
print("Model outputs:", [np.asarray(o).shape for o in outs], f" ({time.time() - t:.2f} s)")
print("Segmentation model (card outlines)" if len(outs) > 1 else "Box model (no outlines)")

## Camera + arm helpers

In [ ]:
def move(q):
    # 6th value is the gripper (held at 90), 1500 = take 1.5 seconds
    arm.Arm_serial_servo_write6(*q, 90, 1500)
    time.sleep(2)


def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def show(frame, dets, title=''):
    img = frame.copy()
    for label, s, (x1, y1, x2, y2) in dets:
        color = (128, 128, 128) if label == 'DOWN' else (0, 255, 0)
        outline = OUTLINES.get((x1, y1, x2, y2))
        if outline is not None:
            cv2.polylines(img, [outline], True, color, 2)            # segmentation: the card's outline
        else:
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{label} {s:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


def read_cards(pose_name):
    move(poses[pose_name])
    frame = take_picture()
    if frame is None:
        print("No camera frame")
        return []
    t = time.time()
    dets = detect(frame)
    labels = [d[0] for d in dets]
    total, soft = hand_value(labels)
    print(f"{pose_name}: {labels}  ->  total {total}{' (soft)' if soft else ''}   [{time.time() - t:.2f} s]")
    show(frame, dets, f"{pose_name}: {labels}  total {total}")
    return labels

## Read the dealer's cards

In [ ]:
dealer = read_cards('dealer')
up = [c for c in dealer if c != 'DOWN']
print("Dealer up card:", up, "| face-down cards:", dealer.count('DOWN'))

## Read my cards

In [ ]:
mine = read_cards('mycards')
total, soft = hand_value(mine)
print("My hand:", mine, "->", total, "| Decision:", "HIT" if total < 17 else "STAND")

## Test loop
Stays at one pose. Change the cards, hit Enter to read again. Type anything + Enter to stop.
Good for checking accuracy: count how many reads are right.

In [ ]:
POSE = 'mycards'

move(poses[POSE])
while input("Enter = read cards, anything else = stop: ") == '':
    frame = take_picture()
    if frame is None:
        print("No camera frame")
        continue
    dets = detect(frame)
    labels = [d[0] for d in dets]
    total, soft = hand_value(labels)
    print(labels, "->", total, "(soft)" if soft else "")
    show(frame, dets, f"{labels}  total {total}")

## Done

In [ ]:
move(poses['home'])
cap.release()